# 20 · The single final held-out check

Runs the check **defined and committed in `19_ensemble_stability.ipynb`, section 6b** (commit d1dd1eb), for the final system: the mean-of-p ensemble of the three original-recipe seeds, thresholds tau(0.5%) = 0.00093, tau(1%) = 0.00235, tau(5%) = 0.01032, with the cannot-tell band. **Executed once.** The code of this notebook was committed before it was run. H4, the pipeline check, comes first and the notebook stops with an error if it fails. The interpretation is **not** in this notebook until the results have been committed; it is added in a separate commit afterwards.

The held-out prompts are not pristine (section 6b preamble): this checks Phase 4 on two more topics and does not replace it.

In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from src.data import load_splits
from src.evaluate import wilson_interval, ClusterSampler
from src.ensemble import PEnsemble, wilson_pct

SEED, N_BOOT = 42, 1000
TAU = {'A': (0.00235, 0.01032), 'B': (0.00093, 0.01032)}          # (flag, cannot-tell upper edge), operating points A and B of section 6b
PROMPTS = {'Electoral college': 'Does the electoral college work?', 'Summer projects': 'Summer projects'}
GENS = ['darragh_claude_v6', 'darragh_claude_v7', 'falcon_180b_v1']
pd.set_option('display.width', 230); pd.set_option('display.max_colwidth', 100)
data = load_splits()
ho = data[data['split'].isin(['heldout_prompt', 'heldout_generator'])].copy()
print(ho.groupby(['split', 'label']).size().to_string())

In [ ]:
# the reference of the p scale: the 2,291 val humans, three seeds (nothing held-out is used for calibration)
vt = data[data['split'] == 'val']
vh = vt[vt['label'] == 0].index
s42_val = pd.read_csv('../results/15_scores.csv', index_col='row_id')['roberta']
f43 = pd.read_csv('../results/17_final/orig_s43_scores.csv', index_col='row_id')['logit']; f44 = pd.read_csv('../results/17_final/orig_s44_scores.csv', index_col='row_id')['logit']
ref = np.stack([s42_val.loc[vh].to_numpy(), f43.loc[vh].to_numpy(), f44.loc[vh].to_numpy()], axis=1)
ens = PEnsemble(ref, 'mean')
print('reference val humans:', ref.shape)

In [ ]:
# score every held-out essay with the three models, same code path (first 512 tokens, fp16), seeds in the order 42, 43, 44
MODELS = {42: '../models/distilroberta_finetuned', 43: '../models/distilroberta_orig_s43', 44: '../models/distilroberta_orig_s44'}
RAW = '../results/20_heldout_raw_logits.csv'
@torch.no_grad()
def score(model_dir, text_list, batch_size=64):
    tok = AutoTokenizer.from_pretrained(model_dir); mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).cuda().eval()
    ids = tok(text_list, truncation=True, max_length=512)['input_ids']
    out = np.empty(len(ids), dtype=np.float32); order = np.argsort([len(x) for x in ids], kind='stable')
    for s in range(0, len(order), batch_size):
        ch = order[s:s + batch_size]; w = max(len(ids[i]) for i in ch)
        x = torch.full((len(ch), w), tok.pad_token_id, dtype=torch.long)
        for r, i in enumerate(ch): x[r, :len(ids[i])] = torch.tensor(ids[i])
        with torch.autocast('cuda', dtype=torch.float16):
            out[ch] = mdl(input_ids=x.cuda(), attention_mask=(x != tok.pad_token_id).long().cuda()).logits.squeeze(1).float().cpu().numpy()
    del mdl; torch.cuda.empty_cache()
    return out
if os.path.exists(RAW):
    raw = pd.read_csv(RAW, index_col='row_id'); assert raw.index.equals(ho.index)
else:
    raw = pd.DataFrame(index=ho.index)
    for sd, mdir in MODELS.items():
        raw[f'logit_s{sd}'] = score(mdir, ho['text'].tolist()); print('scored with seed', sd)
    raw.to_csv(RAW)
ho = ho.join(raw)

## H4 · The pipeline check (run first): seed 42 at the old logit thresholds must reproduce notebook 09

In [ ]:
t09 = pd.read_csv('../results/09_threshold_rates.csv', header=None, skiprows=3)
t09.columns = ['model', 'threshold', 'h_val', 'h_test', 'h_hp', 'h_hg', 'a_val', 'a_test', 'a_hp', 'a_hg', 'p_val', 'p_test', 'p_hp', 'p_hg']
t09 = t09[t09['model'] == 'DistilRoBERTa'].set_index('threshold')
rows, ok = [], True
on = ho[ho['on_topic']]
for name, thr, key in [('"1%"', 4.666, 'FPR 1.0%'), ('"0.5%"', 7.796, 'FPR 0.5%')]:
    hum = on[(on['split'] == 'heldout_prompt') & (on['label'] == 0)]; gen = on[on['split'] == 'heldout_generator']
    k_h, n_h = int((hum['logit_s42'] >= thr).sum()), len(hum); k_g, n_g = int((gen['logit_s42'] >= thr).sum()), len(gen)
    exp_h, exp_g = float(t09.loc[key, 'h_hp']), float(t09.loc[key, 'a_hg'])
    exp_k_h = round(exp_h * n_h / 100); pass_h = abs(k_h - exp_k_h) <= 2; pass_g = abs(100 * k_g / n_g - exp_g) <= 0.1
    ok &= pass_h and pass_g
    rows.append({'threshold': f'{name} = logit {thr}', 'held-out-prompt human false flags %': f'{100 * k_h / n_h:.2f} ({k_h}/{n_h})', 'notebook 09': exp_h, 'within 2 essays': pass_h,
                 'held-out-generator AI caught %': f'{100 * k_g / n_g:.2f} ({k_g}/{n_g})', 'notebook 09 ': exp_g, 'within 0.1 pt': pass_g})
print(pd.DataFrame(rows).set_index('threshold').to_string())
# the new seed-42 logits against the ones of notebook 15 (same model; fp16 noise only)
s15 = pd.read_csv('../results/15_scores.csv', index_col='row_id')['roberta'].loc[ho.index]
print(f"\nmax |new seed-42 logit - notebook 15 logit| over {len(ho):,} held-out essays: {np.abs(ho['logit_s42'] - s15).max():.4f}")
assert ok, 'H4 FAILED: the pipeline does not reproduce notebook 09; STOP and tell the project owner before interpreting anything'
print('\nH4 PASSED')

## The ensemble on the held-out essays: p-values and zones; per-essay scores saved

In [ ]:
X = ho[['logit_s42', 'logit_s43', 'logit_s44']].to_numpy()
ps = ens.p_seeds(X); pe = ens.p_ens(X)
for i, sd in enumerate((42, 43, 44)): ho[f'p_s{sd}'] = ps[:, i]
ho['p_ens'] = pe
def zone(p, flag, band): return np.where(p <= flag, 'flagged', np.where(p <= band, 'cannot tell', 'not flagged'))
for op, (f, b) in TAU.items(): ho[f'zone_{op}'] = zone(pe, f, b)
ho[['split', 'label', 'source', 'prompt_name', 'on_topic', 'logit_s42', 'logit_s43', 'logit_s44', 'p_s42', 'p_s43', 'p_s44', 'p_ens', 'zone_A', 'zone_B']].round(6).to_csv('../results/20_heldout_scores.csv')
print('saved per-essay held-out scores:', len(ho))

## H1 · Human essays of each held-out prompt

In [ ]:
ZONES = ['flagged', 'cannot tell', 'not flagged']
def zone_row(d, op):
    n = len(d); out = {'n': n}
    for z in ZONES:
        k = int((d[f'zone_{op}'] == z).sum()); lo, hi = wilson_interval(k, n)
        out[f'% {z}'] = f'{100 * k / n:.2f} [{100 * lo:.2f}, {100 * hi:.2f}]'
    return out
def boot_zone(d, op, n_boot=N_BOOT):
    sm = ClusterSampler(d['dup_cluster']); rng = np.random.default_rng(SEED); z = d[f'zone_{op}'].to_numpy()
    draws = np.array([[(z[r] == zz).mean() for zz in ZONES] for r in (sm.draw(rng) for _ in range(n_boot))])
    return {f'% {zz}': f'{100 * (z == zz).mean():.2f} [{np.percentile(draws[:, j], 2.5) * 100:.2f}, {np.percentile(draws[:, j], 97.5) * 100:.2f}]' for j, zz in enumerate(ZONES)}, draws
def table(label, extra_filter=None, sources=None):
    rows, pooled_draws = [], {}
    for block, mask in (('all essays', None), ('on-topic only', True)):
        for op in ('A', 'B'):
            pool_frames = []
            if sources is None:
                names = PROMPTS.items(); frames = [(k, ho[(ho['split'] == 'heldout_prompt') & (ho['prompt_name'] == v) & (ho['label'] == label)]) for k, v in names]
            else:
                frames = [(s, ho[(ho['split'] == 'heldout_generator') & (ho['source'] == s)]) for s in sources]
            for name, d in frames:
                d = d[d['on_topic']] if mask else d
                rows.append({'block': block, 'operating point': op, 'rows': name, **zone_row(d, op)}); pool_frames.append(d)
            dp = pd.concat(pool_frames); b, draws = boot_zone(dp, op)
            rows.append({'block': block, 'operating point': op, 'rows': 'POOLED (bootstrap interval)', 'n': len(dp), **b}); pooled_draws[(block, op)] = (dp, draws)
    return pd.DataFrame(rows), pooled_draws
H1, H1_draws = table(0)
H1.set_index(['block', 'operating point', 'rows'])

## H2 · AI essays of the held-out prompts

In [ ]:
H2, H2_draws = table(1)
H2.set_index(['block', 'operating point', 'rows'])

## H3 · AI essays of the held-out generators

In [ ]:
H3, H3_draws = table(1, sources=GENS)
H3.set_index(['block', 'operating point', 'rows'])

## Headline numbers, the failure criterion and H5

In [ ]:
dp, draws = H1_draws[('on-topic only', 'A')]
k_pool = int((dp['zone_A'] == 'flagged').sum()); n_pool = len(dp); rate = 100 * k_pool / n_pool
lo_b, hi_b = np.percentile(draws[:, 0] * 100, [2.5, 97.5]); lo_w, hi_w = (100 * v for v in wilson_interval(k_pool, n_pool))
print(f"HEADLINE 1: pooled ON-TOPIC held-out human essays flagged at operating point A: {rate:.2f}%  ({k_pool}/{n_pool})")
print(f"            cluster-bootstrap 95% interval [{lo_b:.2f}, {hi_b:.2f}] (criterion)   |   Wilson [{lo_w:.2f}, {hi_w:.2f}] (reference)")
print(f"FAILURE CRITERION (declared): 1% target missed iff the bootstrap lower bound > 1%:  lower bound {lo_b:.2f}%  ->  {'MISSED' if lo_b > 1 else 'NOT missed'}")
dg, dgr = H3_draws[('on-topic only', 'A')]
kg = int((dg['zone_A'] == 'flagged').sum()); ng = len(dg)
print(f"HEADLINE 2: pooled ON-TOPIC held-out-generator AI essays caught (flagged) at A: {100 * kg / ng:.2f}%  ({kg}/{ng})  bootstrap [{np.percentile(dgr[:, 0] * 100, 2.5):.2f}, {np.percentile(dgr[:, 0] * 100, 97.5):.2f}]")
# H5: rank and percentile among the 13 left-out-topic rates (rule d, tau(1%), notebook 18b)
loto = pd.read_csv('../results/18b_by_topic_1pct_original.csv'); loto = loto[loto['rule'] == 'd']['false flags %'].to_numpy()
rows = []
for k, v in PROMPTS.items():
    d = ho[(ho['split'] == 'heldout_prompt') & (ho['prompt_name'] == v) & (ho['label'] == 0) & ho['on_topic']]
    r = 100 * (d['zone_A'] == 'flagged').mean()
    rows.append({'held-out prompt': k, 'on-topic humans flagged at A %': round(r, 2), 'rank among 14 (1 = highest)': 1 + int((loto > r).sum()), 'percentile (share of the 13 left-out rates below)': round(100 * (loto < r).mean(), 1),
                 'inside the 13-topic range 0.00-1.74%?': bool(loto.min() <= r <= loto.max()), 'Phase 4: topics over 1% target': '23% (3 of 13)'})
print('\nH5, placement among the 13 left-out-topic rates (rule (d), 1% target; 13 rates from 0.00% to 1.74%):')
pd.DataFrame(rows).set_index('held-out prompt')

## H6 · How much of each zone is AI (operating point A)

In [ ]:
hum = ho[(ho['split'] == 'heldout_prompt') & (ho['label'] == 0) & ho['on_topic']]
ai_all = pd.concat([ho[(ho['split'] == 'heldout_prompt') & (ho['label'] == 1) & ho['on_topic']], ho[(ho['split'] == 'heldout_generator') & ho['on_topic']]])
def shares(h, a):
    return ({z: float((h['zone_A'] == z).mean()) for z in ZONES}, {z: float((a['zone_A'] == z).mean()) for z in ZONES})
def p_ai(h, a, pi): return {z: (pi * a[z] / (pi * a[z] + (1 - pi) * h[z]) if (pi * a[z] + (1 - pi) * h[z]) > 0 else np.nan) for z in ZONES}
PRIORS = {'34.6% (the dataset)': 0.346, '10%': 0.10, '5%': 0.05}
sh, sa = ClusterSampler(hum['dup_cluster']), ClusterSampler(ai_all['dup_cluster']); rng = np.random.default_rng(SEED)
draws = {(z, k): [] for z in ZONES for k in PRIORS}
for _ in range(N_BOOT):
    h, a = shares(hum.iloc[sh.draw(rng)], ai_all.iloc[sa.draw(rng)])
    for k, pi in PRIORS.items():
        for z, v in p_ai(h, a, pi).items(): draws[(z, k)].append(v)
h0, a0 = shares(hum, ai_all)
print('held-out human zone shares %:', {z: round(100 * v, 2) for z, v in h0.items()}, '| AI zone shares %:', {z: round(100 * v, 2) for z, v in a0.items()}, f'| n human {len(hum)}, n AI {len(ai_all)}')
rows = []
for k, pi in PRIORS.items():
    pt = p_ai(h0, a0, pi)
    for z in ZONES:
        d = np.array(draws[(z, k)], dtype=float); d = d[~np.isnan(d)]
        rows.append({'AI share': k, 'zone': z, 'P(AI | zone) %': f'{100 * pt[z]:.1f} [{100 * np.percentile(d, 2.5):.1f}, {100 * np.percentile(d, 97.5):.1f}]'})
H6 = pd.DataFrame(rows); H6.pivot(index='zone', columns='AI share', values='P(AI | zone) %').loc[ZONES][list(PRIORS)]

## H7 · Among held-out humans flagged by the ensemble at A: how many each single seed also flags

In [ ]:
rows = []
for block, mask in (('all essays', False), ('on-topic only', True)):
    frames = [(k, ho[(ho['split'] == 'heldout_prompt') & (ho['prompt_name'] == v) & (ho['label'] == 0)]) for k, v in PROMPTS.items()]
    frames.append(('POOLED', pd.concat([f for _, f in frames])))
    for name, d in frames:
        d = d[d['on_topic']] if mask else d
        fl = d[d['zone_A'] == 'flagged']; t1 = TAU['A'][0]
        rows.append({'block': block, 'rows': name, 'humans': len(d), 'flagged by the ensemble': len(fl), 'also flagged by seed 42': int((fl['p_s42'] <= t1).sum()),
                     'by seed 43': int((fl['p_s43'] <= t1).sum()), 'by seed 44': int((fl['p_s44'] <= t1).sum()),
                     'by all three seeds': int(((fl['p_s42'] <= t1) & (fl['p_s43'] <= t1) & (fl['p_s44'] <= t1)).sum())})
H7 = pd.DataFrame(rows); H7.set_index(['block', 'rows'])

In [ ]:
pd.concat({'H1': H1, 'H2': H2, 'H3': H3, 'H6': H6, 'H7': H7}, names=['table']).to_csv('../results/20_heldout_tables.csv')
print('saved results/20_heldout_scores.csv and results/20_heldout_tables.csv; commit them BEFORE writing any interpretation')